# Урок 14. Базы данных: реляционная модель

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 13](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-13.ipynb) · [Урок 15 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-15.ipynb)

---

Таблицы, поля, типы, ключи. Первичный и внешний ключ. Связи один-ко-многим. Нормализация на практическом уровне. Проектирование схемы.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-14", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Когда таблицы перестаёт хватать

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g11/stoyka.jpg" width="380" alt="Стойка с серверами в дата-центре">

*Стойка с серверами в дата-центре*

<sub>Derrick Coetzee · CC0 · Wikimedia Commons</sub>

Пока данных немного, хватает электронной таблицы. Но представьте
школьную библиотеку, где в одном листе записано всё сразу:

| Читатель | Класс | Телефон | Книга | Автор | Дата выдачи |
|---|---|---|---|---|---|
| Иванов А. | 11А | 900-11-11 | Мастер и Маргарита | Булгаков | 12.09 |
| Иванов А. | 11А | 900-11-11 | Собачье сердце | Булгаков | 20.09 |
| Петрова М. | 10Б | 900-22-22 | Мастер и Маргарита | Булгаков | 25.09 |

Что здесь плохо:

* **избыточность** — телефон Иванова записан дважды, автор книги тоже;
* **аномалии изменения** — сменился телефон, и править надо все
  строки; пропустили одну — данные противоречат сами себе;
* **аномалии удаления** — удалили последнюю выдачу книги и потеряли
  сведения о самой книге;
* **невозможность записать «пока ничего»** — как внести читателя,
  который ещё ничего не брал?

Решение придумал Эдгар Кодд в 1970 году: разложить данные
по нескольким таблицам и связать их ключами. Это **реляционная
модель**, и на ней работают почти все базы данных мира.

### Словарь

| Термин | Что это | В таблице |
|---|---|---|
| таблица (отношение) | набор однотипных записей | лист |
| запись (кортеж) | один объект | строка |
| поле (атрибут) | одна характеристика | столбец |
| тип поля | что можно хранить | INTEGER, TEXT, REAL, DATE |
| первичный ключ | поле, однозначно определяющее запись | обычно `id` |
| внешний ключ | поле, ссылающееся на ключ другой таблицы | `id_читателя` |

### Правила первичного ключа

* он уникален — двух записей с одним ключом быть не может;
* он не пустой;
* он не меняется со временем.

Поэтому телефон или фамилия — плохие ключи: фамилии повторяются,
телефоны меняются. Обычно заводят отдельное числовое поле `id`.

### Разложим библиотеку по таблицам

```
читатели                 книги                    выдачи
┌────┬────────┬──────┐   ┌────┬───────────┬─────┐ ┌────┬────────┬──────┬───────┐
│ id │ фамилия│ класс│   │ id │ название  │автор│ │ id │читатель│ книга│ дата  │
├────┼────────┼──────┤   ├────┼───────────┼─────┤ ├────┼────────┼──────┼───────┤
│ 1  │ Иванов │ 11А  │   │ 1  │ Мастер…   │Булг…│ │ 1  │   1    │  1   │ 12.09 │
│ 2  │ Петрова│ 10Б  │   │ 2  │ Собачье…  │Булг…│ │ 2  │   1    │  2   │ 20.09 │
└────┴────────┴──────┘   └────┴───────────┴─────┘ │ 3  │   2    │  1   │ 25.09 │
                                                   └────┴────────┴──────┴───────┘
```

Телефон читателя теперь хранится ровно в одном месте. Автор книги —
тоже. Связь между таблицами держат числа: в таблице выдач поле
`читатель` — это внешний ключ на `читатели.id`.

### Виды связей

| Связь | Пример | Как реализуется |
|---|---|---|
| один-к-одному | человек — паспорт | внешний ключ с условием уникальности |
| один-ко-многим | читатель — его выдачи | внешний ключ в таблице «многих» |
| многие-ко-многим | книги — жанры | отдельная таблица-связка |

Правило простое: внешний ключ ставится **на стороне «многих»**.
У одного читателя много выдач, поэтому ссылка на читателя лежит
в выдачах, а не наоборот.

Связь «многие-ко-многим» напрямую выразить нельзя: одна книга
в нескольких жанрах, один жанр у многих книг. Заводят третью таблицу
из двух внешних ключей — ровно так же устроена таблица `выдачи`,
которая связывает читателей и книги.

### Нормализация на практическом уровне

Формально есть несколько «нормальных форм», но на школьном уровне
достаточно трёх правил:

1. в одной ячейке — одно значение (не «Иванов, Петров» через запятую);
2. каждый факт хранится ровно в одном месте;
3. поле относится к тому объекту, в таблице которого лежит
   (телефон читателя — в таблице читателей, а не в таблице выдач).

### Целостность

База данных умеет сама следить, чтобы внешний ключ указывал
на существующую запись: попытка выдать книгу несуществующему
читателю будет отклонена. Это называется **ссылочной целостностью**,
и включается она конструкцией `FOREIGN KEY` при создании таблицы.

## Смотрим, как это работает

### Пример 1. Создаём базу прямо в ноутбуке

Модуль `sqlite3` встроен в Python: целая база данных живёт в одном
файле или, как здесь, прямо в памяти.

In [ ]:
import sqlite3

соединение = sqlite3.connect(":memory:")
курсор = соединение.cursor()

курсор.execute("""
CREATE TABLE читатели (
    id      INTEGER PRIMARY KEY,
    фамилия TEXT NOT NULL,
    класс   TEXT
)
""")

курсор.execute("""
CREATE TABLE книги (
    id       INTEGER PRIMARY KEY,
    название TEXT NOT NULL,
    автор    TEXT,
    год      INTEGER
)
""")

курсор.execute("""
CREATE TABLE выдачи (
    id        INTEGER PRIMARY KEY,
    читатель  INTEGER,
    книга     INTEGER,
    дата      TEXT,
    FOREIGN KEY (читатель) REFERENCES читатели (id),
    FOREIGN KEY (книга)    REFERENCES книги (id)
)
""")

print("Таблицы созданы")

`PRIMARY KEY` объявляет первичный ключ, `NOT NULL` запрещает пустое
значение, `FOREIGN KEY … REFERENCES` описывает связь.

### Пример 2. Наполняем данными

In [ ]:
курсор.executemany("INSERT INTO читатели VALUES (?, ?, ?)", [
    (1, "Иванов", "11А"),
    (2, "Петрова", "10Б"),
    (3, "Сидоров", "11А"),
])

курсор.executemany("INSERT INTO книги VALUES (?, ?, ?, ?)", [
    (1, "Мастер и Маргарита", "Булгаков", 1967),
    (2, "Собачье сердце", "Булгаков", 1987),
    (3, "Преступление и наказание", "Достоевский", 1866),
])

курсор.executemany("INSERT INTO выдачи VALUES (?, ?, ?, ?)", [
    (1, 1, 1, "2026-09-12"),
    (2, 1, 2, "2026-09-20"),
    (3, 2, 1, "2026-09-25"),
])

соединение.commit()

for строка in курсор.execute("SELECT * FROM читатели"):
    print(строка)

Вопросительные знаки — места для значений. Подставлять данные прямо
в текст запроса нельзя: это классическая дыра в безопасности,
через которую взламывают сайты.

### Пример 3. Целостность работает

In [ ]:
курсор.execute("PRAGMA foreign_keys = ON")

try:
    курсор.execute("INSERT INTO выдачи VALUES (4, 99, 1, '2026-10-01')")
    соединение.commit()
    print("Запись добавлена")
except sqlite3.IntegrityError as ошибка:
    print("База отклонила запись:", ошибка)

Читателя с номером 99 не существует, и база не дала создать
«висячую» ссылку. Без внешних ключей такая запись спокойно попала бы
в таблицу, и через полгода никто не понял бы, чья это выдача.

### Пример 4. Тот же самый вопрос — к одной таблице и к трём

Разложив данные, мы ничего не потеряли: свести их обратно можно
в любой момент.

In [ ]:
запрос = """
SELECT читатели.фамилия, книги.название, выдачи.дата
FROM выдачи
JOIN читатели ON выдачи.читатель = читатели.id
JOIN книги    ON выдачи.книга    = книги.id
ORDER BY выдачи.дата
"""

for фамилия, название, дата in курсор.execute(запрос):
    print(f"{дата}  {фамилия:<8} {название}")

Это и есть JOIN — соединение таблиц по ключам. Подробно разберём его
через два урока, а пока важно понять главное: разложить данные
не значит их разъединить.

### Пример 5. Что даёт нормализация

In [ ]:
курсор.execute("UPDATE читатели SET класс = '11Б' WHERE id = 1")
соединение.commit()

for строка in курсор.execute("SELECT * FROM читатели WHERE id = 1"):
    print("После перевода в другой класс:", строка)

print("Изменили строк:", курсор.rowcount)

Одна запись — одно изменение. В сплошной таблице пришлось бы менять
столько строк, сколько книг взял Иванов, — и любая пропущенная
сделала бы данные противоречивыми.

## Пробуем сами

### Задача 1. Как называется поле

Как называется поле, однозначно определяющее запись в таблице?

In [ ]:
#@title 🧩 Задача 1. Ключ { display-mode: "form" }
#@markdown Выберите ответ
ключ = "выбери ответ" #@param ["выбери ответ", "первичный ключ", "внешний ключ", "индекс"]

si.ответ("1", ключ, "1ac0b46e443db517",
         hint="Обычно это поле id.")

### Задача 2. Где хранить ссылку

Связь «один учитель — много уроков». В какой таблице должен лежать
внешний ключ?

In [ ]:
#@title 🧩 Задача 2. Сторона связи { display-mode: "form" }
#@markdown Выберите ответ
где_ключ = "выбери ответ" #@param ["выбери ответ", "в таблице уроков", "в таблице учителей", "в обеих"]

si.ответ("2", где_ключ, "d54333a3750f5079",
         hint="Внешний ключ ставится на стороне «многих».")

### Задача 3. Годится ли фамилия в ключи

Можно ли использовать фамилию читателя как первичный ключ?

In [ ]:
#@title 🧩 Задача 3. Фамилия как ключ { display-mode: "form" }
#@markdown Выберите ответ
годится = "выбери ответ" #@param ["выбери ответ", "да", "нет"]

si.ответ("3", годится, "dde7950114f546d0",
         hint="Однофамильцы.")

### Задача 4. Своя база

Напишите функцию, которая создаёт базу в памяти с таблицей `ученики`
(поля `id`, `фамилия`, `класс`), добавляет туда переданные записи
и возвращает их количество.

Записи приходят списком кортежей `(id, фамилия, класс)`.

In [ ]:
def создать_базу(записи):
    return ...

In [ ]:
si.check("4", создать_базу, [
    ([(1, "Иванов", "11А"), (2, "Петрова", "10Б")], 2),
    ([], 0),
    ([(1, "Один", "9В")], 1),
])

### Задача 5. Проверка уникальности ключа

Функция получает список записей (кортежей, первый элемент — id)
и возвращает `True`, если все ключи уникальны.

In [ ]:
def ключи_уникальны(записи):
    return ...

In [ ]:
si.check("5", ключи_уникальны, [
    ([(1, "а"), (2, "б")], True),
    ([(1, "а"), (1, "б")], False),
    ([], True),
])

### Задача 6. Поиск висячих ссылок

Функция получает список записей главной таблицы (кортежи, первый
элемент — id) и список внешних ключей. Возвращает список тех ключей,
для которых записи нет, в порядке появления.

In [ ]:
def висячие(записи, ссылки):
    return ...

In [ ]:
si.check("6", висячие, [
    (([(1, "а"), (2, "б")], [1, 2, 99]), [99]),
    (([(1, "а")], [1, 1, 1]), []),
    (([], [5]), [5]),
])

### Задача 7. Какая связь

«У книги может быть несколько жанров, и в одном жанре много книг».
Какая это связь?

In [ ]:
#@title 🧩 Задача 7. Тип связи { display-mode: "form" }
#@markdown Выберите ответ
связь = "выбери ответ" #@param ["выбери ответ", "один-к-одному", "один-ко-многим", "многие-ко-многим"]

si.ответ("7", связь, "23763156fa0c494a",
         hint="Придётся заводить третью таблицу.")

## Домашнее задание

### Домашнее задание 1. Убираем избыточность

Функция получает список записей вида `(фамилия, класс, книга)`
и возвращает список **уникальных** пар `(фамилия, класс)`,
отсортированный по фамилии. Это первый шаг нормализации: выделение
таблицы читателей.

In [ ]:
def выделить_читателей(записи):
    return ...

In [ ]:
si.check("дз1", выделить_читателей, [
    ([("Иванов", "11А", "Книга1"), ("Иванов", "11А", "Книга2"),
      ("Петрова", "10Б", "Книга1")],
     [("Иванов", "11А"), ("Петрова", "10Б")]),
    ([], []),
])

### Домашнее задание 2. Сборка данных обратно

Функция получает словарь `{id: фамилия}` и список выдач
`(id_читателя, книга)`, возвращает список строк вида
`"Фамилия — Книга"` в исходном порядке. Если читателя нет
в словаре — пропустите выдачу.

In [ ]:
def собрать(читатели, выдачи):
    return ...

In [ ]:
si.check("дз2", собрать, [
    (({1: "Иванов", 2: "Петрова"}, [(1, "Книга1"), (2, "Книга2")]),
     ["Иванов — Книга1", "Петрова — Книга2"]),
    (({1: "Иванов"}, [(99, "Книга1")]), []),
    (({}, []), []),
])

### Домашнее задание 3. Схема под свою задачу

Спроектируйте базу данных для чего-то своего: коллекции игр, секции,
школьного кружка, домашней библиотеки. Нужны минимум три таблицы,
из них хотя бы одна — связка. Для каждой выпишите поля, типы,
первичный ключ и внешние ключи, нарисуйте схему со стрелками
и придумайте пять вопросов, на которые база сможет ответить.
На следующем уроке мы научимся такие вопросы задавать.

---

### Любопытно

Статью 1970 года Кодд назвал «Реляционная модель данных для больших
совместно используемых банков данных». В IBM идею встретили прохладно:
компания уже продавала другую систему, и признавать её устаревшей
никто не хотел. Первую коммерческую реляционную СУБД выпустила
небольшая компания со стороны — сегодня она называется Oracle.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 13](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-13.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 15 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-15.ipynb)